# Selección automática de features con Lasso

Este notebook documenta el proceso de selección de variables automáticas usando Lasso regularizado. Se adapta dinámicamente el nivel de regularización (`alpha`) según el número de features y la naturaleza del problema (regresión o clasificación). Todos los pasos están documentados y se exponen gráficamente los coeficientes y la lista explícita de features seleccionadas.

## 1. Cargar los datos transformados

Supone que `X_train.csv`, `X_test.csv` (features transformados) y `y_train.csv` (target) han sido generados por el Pipeline anterior.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso, LogisticRegression
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import log_loss
import seaborn as sns

# Cargar los features ya transformados
data_path = 'proyecto_pipeline_machine_learning_muestra/'
X_train = pd.read_csv(data_path + 'X_train.csv', index_col=0)
X_test  = pd.read_csv(data_path + 'X_test.csv', index_col=0)
y_train = pd.read_csv(data_path + 'y_train.csv', index_col=0).squeeze()  # Series

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train type:  {y_train.dtype}")

## 2. Preparar el target
- Si es regresión y está sesgado, aplicar `np.log1p` para suavizar outliers.
- Si es clasificación binaria, dejar tal cual.

In [ ]:
# Visualización rápida del target
y_train.hist(bins=50)
plt.title('Distribución del target')
plt.show()

# Identificar automáticamente si se trata de regresión o clasificación
if y_train.nunique() == 2 and set(y_train.unique()) <= {0, 1}:
    problem_type = 'classification'
    print('Detección automática: Clasificación binaria')
else:
    problem_type = 'regression'
    print('Detección automática: Regresión continua')
    # Comprobar sesgo
    skew = y_train.skew()
    print(f'skew={skew:.2f}')
    if np.abs(skew) > 1:
        y_train = np.log1p(y_train)
        print('El target era muy sesgado. Se aplicó np.log1p(y_train) para normalizar.')
    else:
        print('El target tiene un sesgo aceptable.')

## 3. Determinar Alpha/C para Lasso según el número de features
- Si features >100 usar alpha=0.01 (más regularización).
- Si features <=100 usar alpha=0.001 (menos regularización).
- Para clasificación, usar `LogisticRegression(penalty="l1")` con solver adecuado.

In [ ]:
n_features = X_train.shape[1]
if n_features > 100:
    alpha = 0.01
    C = 0.1
else:
    alpha = 0.001
    C = 0.5
print(f"Features: {n_features}, alpha(Lasso): {alpha}, C(LogisticRegression): {C}")

## 4. Ajustar el modelo regularizado

In [ ]:
if problem_type == 'regression':
    # Lasso para regresión
    lasso = Lasso(alpha=alpha, max_iter=2000, random_state=123)
    lasso.fit(X_train, y_train)
    coefs = lasso.coef_
else:
    # Lasso para clasificación (usamos LogisticRegression con L1)
    lasso = LogisticRegression(penalty='l1', C=C, solver='liblinear', random_state=123, max_iter=1000)
    lasso.fit(X_train, y_train)
    coefs = lasso.coef_[0]

# Almacenar los coeficientes como DataFrame para análisis
coef_df = pd.DataFrame({
    'feature': X_train.columns,
    'coef': coefs
})
coef_df['is_selected'] = coef_df['coef'] != 0
coef_df.sort_values('coef', ascending=True, inplace=True)

print(f"Features con coeficiente no cero: {coef_df['is_selected'].sum()} / {n_features}")

## 5. Visualización: Coeficientes de las features

In [ ]:
plt.figure(figsize=(12, int(0.3*n_features)+4))
colors = coef_df['is_selected'].map({True: 'green', False: 'red'})
sns.barplot(
    x='coef',
    y='feature',
    data=coef_df,
    palette=colors.tolist(),
    orient='h',
    dodge=False
)
plt.title('Coeficientes Lasso: Features eliminadas (rojo) y seleccionadas (verde)')
plt.tight_layout()
plt.show()

## 6. Aplicar SelectFromModel para filtrar features

In [ ]:
selector = SelectFromModel(
    lasso, prefit=True, threshold=1e-8 # threshold bajo para que tome coef=0 como corte
)
selected_mask = selector.get_support()
selected_features = X_train.columns[selected_mask].tolist()
removed_features = X_train.columns[~selected_mask].tolist()

# Exportar la lista para siguiente pipeline
pd.Series(selected_features).to_csv(data_path + 'selected_features.csv', index=False)

## 7. Reporte final de selección de features

In [ ]:
print(f"\n--- Reporte de Selección de Features ---")
print(f"Total de features originales:       {n_features}")
print(f"Total de features seleccionadas:     {len(selected_features)}")
print(f"Total de features eliminadas:        {len(removed_features)}")
print(f"\nLista de features seleccionadas (top 20):\n{selected_features[:20]}")

## 8. Exportar lista de features seleccionadas

La lista exportada está disponible en `proyecto_pipeline_machine_learning_muestra/selected_features.csv` para ser utilizada en pasos subsiguientes del pipeline de machine learning.

In [ ]:
# (Ya exportado arriba, solo mostrar ruta y preview)
print(f"Archivo guardado: {data_path + 'selected_features.csv'}")
print(pd.read_csv(data_path + 'selected_features.csv').head())

---

# Conclusión

El proceso automático de selección de variables vía Lasso permite filtrar las variables más relevantes completamente agnóstico al dominio, optimizando la generalización del modelo en la siguiente etapa.